<a href="https://colab.research.google.com/github/LucasMartinscode/LucasMartinscode/blob/LucasMartinscode-patch-1/Aula07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Aula 07- exercicios

#Diferencial

In [ ]:
import pandas as pd
from faker import Faker
from datetime import datetime
import random
import os
import time

fake = Faker("pt_BR")

output_file = r'C:\Users\LucasMartins\OneDrive - SUPERCAMPO S.A\Desktop\Eng de dados\awari-engenharia-de-dados-docker-main\awari-engenharia-de-dados-docker-main\exercicios\municipios-estados\streaming\estados.csv'

if not os.path.exists(os.path.dirname(output_file)):
    os.makedirs(os.path.dirname(output_file))

# Verifica se o arquivo já existe
file_exists = os.path.isfile(output_file)

for e in range(10):  # Gerando 10 blocos de dados para teste
    lista_de_usuarios = []

    for _ in range(random.randint(1, 10)):  # Gerando entre 1 e 10 registros por bloco
        data = {
            'codigo_uf': fake.unique.random_int(min=1, max=99999999),
            'uf': fake.state_abbr(),
            'nome': fake.name(),
            'latitude': fake.latitude(),
            'longitude': fake.longitude(),
            'regiao': fake.state()
        }
        lista_de_usuarios.append(data)

    df = pd.DataFrame(lista_de_usuarios)

    if file_exists:
        df.to_csv(output_file, index=False, header=False, mode='a')  # Adiciona sem cabeçalho
    else:
        df.to_csv(output_file, index=False, header=True, mode='a')  # Adiciona com cabeçalho na primeira vez
        file_exists = True  # Atualiza o estado para não incluir cabeçalho nas próximas vezes

    print(f'Dados adicionados ao arquivo: {output_file}')

    time.sleep(random.randint(1, 2))  # Dorme por 1 a 2 segundos


# Kafka Streaming


-- producer

In [ ]:
import pandas as pd
from faker import Faker
from datetime import datetime
import random
import os
import time
from json import dumps
from kafka import KafkaProducer

fake = Faker("pt_BR")

output_dir = r'C:\Users\LucasMartins\OneDrive - SUPERCAMPO S.A\Desktop\Eng de dados\awari-engenharia-de-dados-docker-main\awari-engenharia-de-dados-docker-main\exercicios\municipios-estados\streaming'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Cria um producer, responsavel por enviar mensagens ao Kafka
producer = KafkaProducer(bootstrap_servers=['awari-kafka:9093'],
                         value_serializer=lambda x:
                         dumps(x).encode('utf-8'))

def gerar_dados():
    lista_de_usuarios = []
    for _ in range(random.randint(1, 10)):  # Gerando entre 1 e 10 registros por bloco
        data = {
            'codigo_uf': fake.unique.random_int(min=1, max=99999999),
            'uf': fake.state_abbr(),
            'nome': fake.name(),
            'latitude': fake.latitude(),
            'longitude': fake.longitude(),
            'regiao': fake.state()
        }
        lista_de_usuarios.append(data)
    return lista_de_usuarios

for e in range(10):  # Gerando 10 blocos de dados para teste
    dados = gerar_dados()
    producer.send('municipios', value=dados)
    producer.flush()  # Assegura que os dados foram enviados
    print(f'Dados enviados para o Kafka: {dados}')
    time.sleep(random.randint(1, 2))  # Dorme por 1 a 2 segundos


-consumer

In [ ]:
import os
import pandas as pd
from kafka import KafkaConsumer
from json import loads

output_file = r'C:\Users\LucasMartins\OneDrive - SUPERCAMPO S.A\Desktop\Eng de dados\awari-engenharia-de-dados-docker-main\awari-engenharia-de-dados-docker-main\exercicios\municipios-estados\streaming\estados.csv'

if not os.path.exists(os.path.dirname(output_file)):
    os.makedirs(os.path.dirname(output_file))

# Verifica se o arquivo já existe
file_exists = os.path.isfile(output_file)

consumer = KafkaConsumer(
    'municipios',
    bootstrap_servers=['awari-kafka:9093'],
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='my-group',
    value_deserializer=lambda x: loads(x.decode('utf-8'))
)

for message in consumer:
    lista_de_usuarios = message.value

    df = pd.DataFrame(lista_de_usuarios)

    if file_exists:
        df.to_csv(output_file, index=False, header=False, mode='a')  # Adiciona sem cabeçalho
    else:
        df.to_csv(output_file, index=False, header=True, mode='a')  # Adiciona com cabeçalho na primeira vez
        file_exists = True  # Atualiza o estado para não incluir cabeçalho nas próximas vezes

    print(f'Dados adicionados ao arquivo: {output_file}')
